# Fine-tune Khmer XLS-R on 2,000 rows in Colab

This follow-up keeps all 1,000 rows and split assignments from the pilot, then adds 1,000 new rows. The same 126 test rows stay held out. The final 2,000-row total includes those 126 test rows.

The model starts from `vitouphy/wav2vec2-xls-r-300m-khmer`, the same pretrained checkpoint used for the pilot. The notebook saves the new manifest, materialized audio, reports, and resumable training checkpoints in Google Drive.

**Hugging Face access is required.** This dataset is gated. Your Hugging Face account must have accepted the dataset conditions, and the notebook needs a Hugging Face read token entered in the masked access-check field. If your access request is still pending, switching from Kaggle to Colab will not grant access; the manifest step will stop with an access error.


## 1. Upload the source files, then organize them

Upload these individual files from the VS Code project to the Colab Contents root. Then run the next cell once; it creates `/content/scripts` and `/content/khmer_transliteration` if needed and moves the files into the right folders.

- `requirements-asr.txt`
- `scripts/build_asr_manifest.py`
- `scripts/finetune_asr.py`
- `scripts/run_asr_baseline.py`
- `scripts/score_asr_report.py`
- `khmer_transliteration/__init__.py`
- `khmer_transliteration/asr_audio.py`
- `khmer_transliteration/asr_data.py`
- `khmer_transliteration/asr_decoding.py`
- `khmer_transliteration/asr_inference.py`
- `khmer_transliteration/asr_metrics.py`
- `khmer_transliteration/asr_text.py`
- `khmer_transliteration/paths.py`

In [7]:
from pathlib import Path
import shutil

CONTENT_DIR = Path("/content")
UPLOADS = {
    "requirements-asr.txt": Path("requirements-asr.txt"),
    "build_asr_manifest.py": Path("scripts/build_asr_manifest.py"),
    "finetune_asr.py": Path("scripts/finetune_asr.py"),
    "run_asr_baseline.py": Path("scripts/run_asr_baseline.py"),
    "score_asr_report.py": Path("scripts/score_asr_report.py"),
    "__init__.py": Path("khmer_transliteration/__init__.py"),
    "asr_audio.py": Path("khmer_transliteration/asr_audio.py"),
    "asr_data.py": Path("khmer_transliteration/asr_data.py"),
    "asr_decoding.py": Path("khmer_transliteration/asr_decoding.py"),
    "asr_inference.py": Path("khmer_transliteration/asr_inference.py"),
    "asr_metrics.py": Path("khmer_transliteration/asr_metrics.py"),
    "asr_text.py": Path("khmer_transliteration/asr_text.py"),
    "paths.py": Path("khmer_transliteration/paths.py"),
}

CONTENT_DIR.mkdir(parents=True, exist_ok=True)
for folder in ("scripts", "khmer_transliteration"):
    (CONTENT_DIR / folder).mkdir(parents=True, exist_ok=True)

# Check every upload first so a missing file does not partially organize the set.
sources = {}
missing = []
for filename, relative_destination in UPLOADS.items():
    destination = CONTENT_DIR / relative_destination
    uploaded_at_root = CONTENT_DIR / filename
    if destination.is_file():
        sources[filename] = destination
    elif uploaded_at_root.is_file():
        sources[filename] = uploaded_at_root
    else:
        missing.append(filename)

if missing:
    missing_list = "\n".join(f"  - {name}" for name in missing)
    raise FileNotFoundError(
        "Upload these individual files to the Colab Contents root, then rerun this cell:\n"
        + missing_list
    )

for filename, relative_destination in UPLOADS.items():
    source = sources[filename]
    destination = CONTENT_DIR / relative_destination
    if source != destination:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(source), str(destination))
    print("Ready:", destination)

print(f"Organized all {len(UPLOADS)} required files under /content.")


Ready: /content/requirements-asr.txt
Ready: /content/scripts/build_asr_manifest.py
Ready: /content/scripts/finetune_asr.py
Ready: /content/scripts/run_asr_baseline.py
Ready: /content/scripts/score_asr_report.py
Ready: /content/khmer_transliteration/__init__.py
Ready: /content/khmer_transliteration/asr_audio.py
Ready: /content/khmer_transliteration/asr_data.py
Ready: /content/khmer_transliteration/asr_decoding.py
Ready: /content/khmer_transliteration/asr_inference.py
Ready: /content/khmer_transliteration/asr_metrics.py
Ready: /content/khmer_transliteration/asr_text.py
Ready: /content/khmer_transliteration/paths.py
Organized all 13 required files under /content.


In [8]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Colab before continuing.")
print("GPU:", torch.cuda.get_device_name(0))


Python: /usr/bin/python3
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


In [9]:
%pip install -q -r /content/requirements-asr.txt ipywidgets


## 2. Connect Google Drive and restore the pilot files

The manifest builder needs the existing `voice_1000` CSVs. This cell restores them from Drive when the Colab runtime starts empty.

In [10]:
from google.colab import drive
from pathlib import Path
import json
import shutil
import subprocess
import sys

drive.mount("/content/drive")

BASE_MANIFEST_DIR = Path("/content/data/voice_1000")
DRIVE_ROOT = Path("/content/drive/MyDrive/khmer_asr")
DRIVE_BASE_MANIFEST_DIR = DRIVE_ROOT / "voice_1000"
MANIFEST_DIR = Path("/content/data/voice_2000")
MATERIALIZED_DIR = MANIFEST_DIR / "materialized_training"
MODEL_DIR = DRIVE_ROOT / "models/khmer_xlsr_ddd_2000_clean_resumable"
REPORT_PATH = MANIFEST_DIR / "finetuned_test_2000_clean.json"
DRIVE_MANIFEST_DIR = DRIVE_ROOT / "voice_2000"
DRIVE_MATERIALIZED_DIR = DRIVE_MANIFEST_DIR / "materialized_training"

# Set True after reconnecting only when a compatible checkpoint exists on Drive.
RESUME_TRAINING = False

if not (BASE_MANIFEST_DIR / "manifest_summary.json").is_file():
    if not (DRIVE_BASE_MANIFEST_DIR / "manifest_summary.json").is_file():
        raise FileNotFoundError(
            "The prior voice_1000 manifest was not found in /content or "
            "/content/drive/MyDrive/khmer_asr/voice_1000."
        )
    BASE_MANIFEST_DIR.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_BASE_MANIFEST_DIR, BASE_MANIFEST_DIR, dirs_exist_ok=True)

if not (MANIFEST_DIR / "manifest_summary.json").is_file() and (
    DRIVE_MANIFEST_DIR / "manifest_summary.json"
).is_file():
    MANIFEST_DIR.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_MANIFEST_DIR, MANIFEST_DIR, dirs_exist_ok=True)

if not (MATERIALIZED_DIR / "materialization_summary.json").is_file() and (
    DRIVE_MATERIALIZED_DIR / "materialization_summary.json"
).is_file():
    MATERIALIZED_DIR.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_MATERIALIZED_DIR, MATERIALIZED_DIR, dirs_exist_ok=True)
    print("Restored materialized audio and manifests from Drive.")

for split in ("train", "validation", "test"):
    if not (BASE_MANIFEST_DIR / f"{split}.csv").is_file():
        raise FileNotFoundError(f"The existing 1,000-row manifest is missing {split}.csv")

PROJECT_ROOT = Path("/content")
def run_project_script(script, *arguments):
    command = [sys.executable, str(PROJECT_ROOT / "scripts" / script), *map(str, arguments)]
    print("Running:", " ".join(command), flush=True)
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

print("Existing pilot manifest:", BASE_MANIFEST_DIR)
print("2,000-row output manifest:", MANIFEST_DIR)
print("Drive checkpoint folder:", MODEL_DIR)
print("Resume mode:", RESUME_TRAINING)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Existing pilot manifest: /content/data/voice_1000
2,000-row output manifest: /content/data/voice_2000
Drive checkpoint folder: /content/drive/MyDrive/khmer_asr/models/khmer_xlsr_ddd_2000_clean_resumable
Resume mode: False


## 3. Verify Hugging Face access

The VS Code Colab extension cannot retrieve Colab Secrets with `userdata.get()`. This cell uses a masked password field instead. Paste your Hugging Face read token into the field and click **Check token and dataset access**. The field clears after the check; the token is not written into the notebook source.

This checks two separate things: that the token identifies your Hugging Face account, and that the account has access to the gated dataset.

In [11]:
import os
import ipywidgets as widgets
from IPython.display import clear_output, display
from huggingface_hub import HfApi, login, whoami

DATASET_ID = "Digital-Divide-Data/khmer-speech-dataset"
needs_hf_access = not (
    (MANIFEST_DIR / "manifest_summary.json").is_file()
    and (MATERIALIZED_DIR / "materialization_summary.json").is_file()
)
HF_DATASET_ACCESS_CONFIRMED = not needs_hf_access

if needs_hf_access:
    token_field = widgets.Password(
        description="HF read token:",
        placeholder="Paste token here",
        layout=widgets.Layout(width="520px"),
    )
    check_button = widgets.Button(
        description="Check token and dataset access",
        button_style="primary",
    )
    access_output = widgets.Output()

    def check_hf_access(_):
        global HF_DATASET_ACCESS_CONFIRMED
        hf_token = token_field.value.strip()
        token_field.value = ""
        with access_output:
            clear_output()
            if not hf_token:
                print("Paste your Hugging Face read token into the masked field first.")
                return
            try:
                account = whoami(token=hf_token)["name"]
            except Exception as exc:
                HF_DATASET_ACCESS_CONFIRMED = False
                print(f"The token could not authenticate: {type(exc).__name__}: {exc}")
                print("Check that it is a valid Hugging Face read token.")
                return
            try:
                HfApi(token=hf_token).dataset_info(DATASET_ID)
            except Exception as exc:
                HF_DATASET_ACCESS_CONFIRMED = False
                print(f"Token is valid for @{account}, but dataset access was denied: {type(exc).__name__}: {exc}")
                print("Accept the dataset conditions and wait for approval if the request is pending, then run this check again.")
                return
            login(token=hf_token, add_to_git_credential=False)
            os.environ["HF_TOKEN"] = hf_token
            HF_DATASET_ACCESS_CONFIRMED = True
            print(f"Token authenticated as @{account}.")
            print("Dataset access confirmed. You can continue to the manifest cell.")

    check_button.on_click(check_hf_access)
    display(widgets.VBox([token_field, check_button, access_output]))
else:
    print("Found the complete 2,000-row manifest and materialized audio in Drive; remote dataset access is not needed for this resumed run.")


## 4. Build the 2,000-row manifest

This keeps the pilot's existing 793 train, 81 validation, and 126 test rows, then adds 1,000 new non-test rows. Existing test speakers and transcript groups stay out of training and validation.


In [12]:
if needs_hf_access and not HF_DATASET_ACCESS_CONFIRMED:
    raise RuntimeError("Complete the Hugging Face token and dataset-access check above before building new rows.")

summary_path = MANIFEST_DIR / "manifest_summary.json"
if summary_path.is_file():
    manifest_summary = json.loads(summary_path.read_text(encoding="utf-8"))
    if manifest_summary.get("selected_rows") != 2000 or manifest_summary.get("added_rows") != 1000:
        raise RuntimeError("The existing voice_2000 manifest is incomplete or has the wrong row count; remove its local and Drive copies before rebuilding.")
    print("Reusing existing 2,000-row manifest:", MANIFEST_DIR)
else:
    run_project_script(
        "build_asr_manifest.py",
        "--target-rows", "2000",
        "--scan-rows", "10000",
        "--extend-manifest-dir", BASE_MANIFEST_DIR,
        "--output-dir", MANIFEST_DIR,
    )
    manifest_summary = json.loads(summary_path.read_text(encoding="utf-8"))

print("Total rows:", manifest_summary["selected_rows"])
print("New rows added:", manifest_summary["added_rows"])
print("Split counts:", manifest_summary["split_counts"])
print("Group overlap:", manifest_summary["group_overlap"])


RuntimeError: Complete the Hugging Face token and dataset-access check above before building new rows.

In [ ]:
from khmer_transliteration.asr_data import read_manifest_rows

base_rows_by_split = {
    split: read_manifest_rows(BASE_MANIFEST_DIR, split)
    for split in ("train", "validation", "test")
}
new_rows_by_split = {
    split: read_manifest_rows(MANIFEST_DIR, split)
    for split in ("train", "validation", "test")
}
assert sum(map(len, new_rows_by_split.values())) == 2000
assert sum(len(rows) for split, rows in new_rows_by_split.items() if split != "test") - sum(len(rows) for split, rows in base_rows_by_split.items() if split != "test") == 1000
for split in ("train", "validation", "test"):
    old_by_index = {row["source_row_index"]: row for row in base_rows_by_split[split]}
    new_by_index = {row["source_row_index"]: row for row in new_rows_by_split[split]}
    assert all(new_by_index.get(index) == row for index, row in old_by_index.items()), f"Existing {split} rows changed or moved"
assert base_rows_by_split["test"] == new_rows_by_split["test"], "Held-out test rows changed"
for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    left_speakers = {row["speaker_id"] for row in new_rows_by_split[left]}
    right_speakers = {row["speaker_id"] for row in new_rows_by_split[right]}
    assert not left_speakers & right_speakers, f"Speaker overlap: {left}/{right}"
    left_groups = {row["group_key"] for row in new_rows_by_split[left]}
    right_groups = {row["group_key"] for row in new_rows_by_split[right]}
    assert not left_groups & right_groups, f"Transcript-group overlap: {left}/{right}"
print("Verified: all 1,000 pilot rows retained, 1,000 new rows added, test set unchanged, and no speaker/group leakage.")


In [ ]:
DRIVE_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
shutil.copytree(MANIFEST_DIR, DRIVE_MANIFEST_DIR, dirs_exist_ok=True)
print("Saved 2,000-row manifests to Drive:", DRIVE_MANIFEST_DIR)


## 5. Materialize the audio

This downloads the audio referenced by the 2,000-row manifest and writes WAV files and training manifests. It requires Hugging Face dataset access but does not use GPU training time.


In [ ]:
materialized_summary_path = MATERIALIZED_DIR / "materialization_summary.json"
materialized_manifests_exist = all(
    (MATERIALIZED_DIR / "manifests" / f"{split}.jsonl").is_file()
    for split in ("train", "validation", "test")
)
if materialized_summary_path.is_file() and materialized_manifests_exist:
    print("Reusing existing materialized 2,000-row audio:", MATERIALIZED_DIR)
else:
    run_project_script(
        "finetune_asr.py",
        "--stage", "materialize",
        "--manifest-dir", MANIFEST_DIR,
        "--materialized-dir", MATERIALIZED_DIR,
    )

shutil.copytree(MATERIALIZED_DIR, DRIVE_MATERIALIZED_DIR, dirs_exist_ok=True)
materialization_summary = json.loads(
    materialized_summary_path.read_text(encoding="utf-8")
)
print("Materialized split counts:", materialization_summary["counts"])
print("Audio and manifests backed up to Drive.")


## 6. Smoke training

Run this short training first. Continue to full training only if it completes successfully.


In [ ]:
SMOKE_DIR = Path("/content/models/khmer_xlsr_ddd_2000_smoke")
if RESUME_TRAINING:
    print("Resume mode: skipping smoke training.")
else:
    run_project_script(
        "finetune_asr.py",
        "--stage", "smoke",
        "--manifest-dir", MANIFEST_DIR,
        "--materialized-dir", MATERIALIZED_DIR,
        "--output-dir", SMOKE_DIR,
    )


## 7. Full training

Full training uses the train and validation rows for three epochs; the test split remains untouched. Checkpoints are written to Google Drive every 250 optimizer steps.

If Colab disconnects after a checkpoint is saved, reconnect and run the setup cells. They restore the 2,000-row manifest and materialized WAVs from Drive. Set `RESUME_TRAINING = True` in the Drive/path cell, then run the smoke and full-training cells; smoke training will be skipped and the full run will resume from the compatible Drive checkpoint. The saved model and training metadata must remain in the same Drive folder.


In [ ]:
full_training_args = [
    "--stage", "full",
    "--manifest-dir", MANIFEST_DIR,
    "--materialized-dir", MATERIALIZED_DIR,
    "--output-dir", MODEL_DIR,
    "--checkpoint-steps", "250",
]
if RESUME_TRAINING:
    full_training_args.append("--resume")
run_project_script("finetune_asr.py", *full_training_args)


## 8. Evaluate on the unchanged test set


In [ ]:
run_project_script(
    "run_asr_baseline.py",
    "--model", MODEL_DIR,
    "--manifest-dir", MANIFEST_DIR,
    "--split", "test",
    "--max-samples", "0",
    "--output", REPORT_PATH,
)
run_project_script(
    "score_asr_report.py",
    "--report", REPORT_PATH,
    "--output", MANIFEST_DIR / "finetuned_test_2000_clean_rescored.json",
)

rescored_path = MANIFEST_DIR / "finetuned_test_2000_clean_rescored.json"
report = json.loads(rescored_path.read_text(encoding="utf-8"))
metrics = report["metrics"]
print("2,000-row model:", report["model"])
print("Test samples:", report.get("successful_samples"))
print(f"CER: {metrics['cer'] * 100:.2f}%")
print(f"Khmer-aware WER: {metrics['wer'] * 100:.2f}%")
print(f"Mean latency: {metrics['mean_latency_seconds']:.3f} seconds")

for name in (REPORT_PATH.name, rescored_path.name):
    shutil.copy2(MANIFEST_DIR / name, DRIVE_MANIFEST_DIR / name)
print("Saved evaluation reports to Drive:", DRIVE_MANIFEST_DIR)
